In [1]:
import lfox
import lfox.lattice as lat
import jax
import jax.numpy as jnp

import numpy as np

In [2]:
d = 2
L = 32

In [3]:
MyLat = lat.SquareLattice(dims=((L,)*d))
MyLat

In [4]:
phi_field = lat.LatticeField(MyLat)
phi_field.field = np.ones_like(phi_field.field)
phi_field.field[0,0] = 0.
phi_field.field = jnp.array(phi_field.field)
print(phi_field.field, phi_field)

[[0. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]] <lfox.lattice.LatticeField object at 0x17786db50>


In [5]:
phi_field.nn_field(0)

Array([[1., 1., 1., ..., 1., 1., 1.],
       [0., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.]], dtype=float32)

In [6]:
p2 = phi_field.copy()
p2 += phi_field
p2.field

Array([[0., 2., 2., ..., 2., 2., 2.],
       [2., 2., 2., ..., 2., 2., 2.],
       [2., 2., 2., ..., 2., 2., 2.],
       ...,
       [2., 2., 2., ..., 2., 2., 2.],
       [2., 2., 2., ..., 2., 2., 2.],
       [2., 2., 2., ..., 2., 2., 2.]], dtype=float32)

In [7]:
print(p2)
p2 += phi_field
print(p2, p2.field)

<lfox.lattice.LatticeField object at 0x11ee601d0> [[0. 3. 3. ... 3. 3. 3.]
 [3. 3. 3. ... 3. 3. 3.]
 [3. 3. 3. ... 3. 3. 3.]
 ...
 [3. 3. 3. ... 3. 3. 3.]
 [3. 3. 3. ... 3. 3. 3.]
 [3. 3. 3. ... 3. 3. 3.]]


In [8]:
@jax.jit
def scalar_action(phi):
    S = 0.0
    F = phi.field
    for ax in range(d):
        S -= 2 * jnp.sum(F * phi.nn_field(ax))
    phi2 = F**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2 )

    return S

In [9]:
sum(jnp.array([1,2]), jnp.array([3,4]))

Array([6, 7], dtype=int32)

In [10]:
%time scalar_action(phi_field)

CPU times: user 22.9 ms, sys: 4.06 ms, total: 27 ms
Wall time: 22.4 ms


Array(-3064., dtype=float32)

In [11]:
import lfox.evolution.hmc as lhmc

In [12]:
itest = lhmc.LeapfrogIntegrator(0.005, 200)
print(itest)

In [13]:
class ScalarAction(lhmc.Action):
    @staticmethod
    def _Sjax(phi):
        return scalar_action(phi)

    def _compute_forces(self):
        self.grads = {
            'phi': jax.jit(jax.grad(scalar_action)),
#            'phi': (jax.grad(scalar_action)),
        }

        def force_func(fields):
            return {'phi': self.grads['phi'](fields['phi']) }

        self.forces = force_func



S2test = ScalarAction({'phi': phi_field}, params={'lamb': 1.1})

print(S2test)

In [14]:
%time S2test.S()

CPU times: user 4.7 ms, sys: 766 µs, total: 5.47 ms
Wall time: 4.61 ms


Array(-3064., dtype=float32)

In [15]:
F2test = S2test.get_forces(recompute=False)

In [16]:
gg = jax.grad(scalar_action)

In [17]:
%time gg(phi_field)

CPU times: user 78.3 ms, sys: 7.77 ms, total: 86 ms
Wall time: 80.2 ms


In [18]:
%time F2test(S2test.fields)

CPU times: user 47 ms, sys: 6.46 ms, total: 53.4 ms
Wall time: 45.6 ms


{'phi': <lfox.lattice.LatticeField at 0x282a4cb10>}

In [19]:
HMCTest = lhmc.HMCEvolver(S2test, 0, itest)
print(HMCTest)

In [23]:
%%time

for _ in range(10):
    HMCTest.evolve(warmup=True)

for _ in range(10):
    HMCTest.evolve()

CPU times: user 4.31 s, sys: 711 ms, total: 5.03 s
Wall time: 4.03 s


In [31]:
%%prun -s cumulative
HMCTest.evolve()

         4229 function calls (4205 primitive calls) in 0.010 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.010    0.010 {built-in method builtins.exec}
        1    0.000    0.000    0.010    0.010 <string>:1(<module>)
        1    0.002    0.002    0.010    0.010 hmc.py:306(evolve)
        1    0.000    0.000    0.005    0.005 hmc.py:274(mom_refresh)
       14    0.000    0.000    0.004    0.000 core.py:388(bind_with_trace)
        2    0.000    0.000    0.004    0.002 traceback_util.py:162(reraise_with_filtered_traceback)
        2    0.000    0.000    0.004    0.002 pjit.py:251(cache_miss)
       14    0.000    0.000    0.003    0.000 core.py:820(process_primitive)
        2    0.000    0.000    0.003    0.002 pjit.py:160(_python_pjit_helper)
        1    0.000    0.000    0.003    0.003 random.py:621(normal)
       12    0.000    0.000    0.002    0.000 core.py:383(bind)
        2   

In [50]:
print(HMCTest.monitor['P_acc'])

[Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(3.831008e+22, dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), Array(0., dtype=float32), A

In [28]:
import matplotlib.pyplot as plt

plt.plot(HMCTest.monitor['delta_H'])

ModuleNotFoundError: No module named 'matplotlib'

In [30]:
print(HMCTest.field_chain['phi'][0].field)
print(HMCTest.action.fields['phi'].field)

[[[[1.4750221 1.7144667 1.8661851 ... 1.4672127 1.4625605 1.6385411]
   [1.748442  1.7631197 1.8845245 ... 1.4374543 1.5245452 1.7336133]
   [1.967168  1.9769256 2.10954   ... 2.0747333 1.5697951 1.5686536]
   ...
   [2.168827  1.5181215 1.3808346 ... 2.014403  2.2026744 1.9167013]
   [1.5925506 1.4156674 1.4523479 ... 1.7474117 1.8128371 1.762194 ]
   [1.6036265 1.9882123 1.9418365 ... 1.6773881 1.7731488 2.1960185]]

  [[1.7861493 1.4507365 1.7084843 ... 1.6716641 1.699941  1.672612 ]
   [1.5891985 1.5235537 1.6217769 ... 1.6284562 1.5484858 1.5523458]
   [1.5635351 1.6134456 1.6187131 ... 1.5705779 1.5649112 1.7895067]
   ...
   [1.6105022 1.4322047 1.5645676 ... 1.5824101 1.8879995 1.93324  ]
   [1.7111028 1.6846585 1.4230918 ... 1.7970833 1.6011578 1.599969 ]
   [1.7646284 1.7449466 1.7483258 ... 1.6347289 1.7187904 1.5345942]]

  [[1.4505941 1.444772  1.5388263 ... 1.3167337 1.3160939 1.8753111]
   [1.2488574 1.6980067 1.4521904 ... 1.375059  1.4922366 1.3554267]
   [1.4517537 1.